# EDA — dữ liệu landing: OpenDengue + Google News RSS

Mục tiêu: hiểu nhanh dữ liệu **raw** (trước Bronze) có những gì, chất lượng ra sao, để quyết định cách làm sạch ở Silver.

File cần upload (lấy từ `data/landing/` trong repo):

| Nguồn | File |
|---|---|
| OpenDengue | `opendengue/<ngày>/National_extract_V1_3.csv` |
| Google News RSS | `news_rss/<ngày>/google_news_*.jsonl` (+ `.xml` nếu muốn xem metadata feed) |

Có 2 cách đưa dữ liệu vào Colab — chọn một ở cell bên dưới:
1. **Upload trực tiếp** (`files.upload()`), chọn cùng lúc csv + jsonl + xml.
2. **Google Drive**: copy nguyên thư mục `landing/` lên Drive rồi sửa `DRIVE_LANDING`.

In [ ]:
# Chọn cách nạp dữ liệu: "upload" | "drive" | "local"
MODE = "upload"
DRIVE_LANDING = "/content/drive/MyDrive/aio-topic-team-pyspark/data/landing"
LOCAL_LANDING = "../data/landing"  # khi chạy notebook ngay trong repo

from pathlib import Path

if MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    DATA_DIR = Path("/content")
elif MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_LANDING)
else:
    DATA_DIR = Path(LOCAL_LANDING)

# Tìm file đệ quy nên cấu trúc thư mục nào cũng được
dengue_files = sorted(DATA_DIR.rglob("National_extract*.csv"))
news_files = sorted(DATA_DIR.rglob("google_news*.jsonl"))
xml_files = sorted(DATA_DIR.rglob("google_news*.xml"))
print("OpenDengue:", dengue_files)
print("News jsonl:", news_files)
print("News xml  :", xml_files)
assert dengue_files and news_files, "Chưa tìm thấy file — kiểm tra lại bước upload / đường dẫn"

In [ ]:
import re
import json
import html
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

---
# Phần 1 — OpenDengue (National extract V1.3)

Mỗi dòng = số ca dengue của **1 quốc gia** trong **1 khoảng thời gian** (`calendar_start_date` → `calendar_end_date`).

In [ ]:
# Đọc toàn bộ là string trước để thấy đúng giá trị raw (vd. "NA"), sau đó mới ép kiểu
raw = pd.read_csv(dengue_files[-1], dtype=str, keep_default_na=False)
print(raw.shape)
raw.head()

## 1.1 Schema, null, số giá trị unique

In [ ]:
def profile(df):
    out = pd.DataFrame({
        "n_unique": df.nunique(),
        "n_NA_string": (df == "NA").sum(),
        "n_empty": (df == "").sum(),
        "sample": df.iloc[0],
    })
    out["pct_NA"] = (out.n_NA_string / len(df) * 100).round(1)
    return out

profile(raw)

> Để ý: các cột toàn `"NA"` (100%) là cột **không có thông tin** ở extract cấp quốc gia (Admin0) → có thể bỏ ở Silver.
> Null ở đây được encode bằng chuỗi `"NA"` (kiểu R), không phải ô trống.

In [ ]:
# Ép kiểu để phân tích
df = raw.replace("NA", np.nan).copy()
df["calendar_start_date"] = pd.to_datetime(df["calendar_start_date"], errors="coerce")
df["calendar_end_date"] = pd.to_datetime(df["calendar_end_date"], errors="coerce")
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["dengue_total"] = pd.to_numeric(df["dengue_total"], errors="coerce")
df["period_days"] = (df["calendar_end_date"] - df["calendar_start_date"]).dt.days + 1

all_null_cols = [c for c in df.columns if df[c].isna().all()]
print("Cột 100% null:", all_null_cols)
df.dtypes

## 1.2 Các cột phân loại

In [ ]:
for col in ["S_res", "T_res", "case_definition_standardised"]:
    print(f"--- {col}")
    print(df[col].value_counts(dropna=False).to_string(), "\n")

> `case_definition_standardised` có cả `Confirmed` và `confirmed` → **không nhất quán hoa/thường**, cần chuẩn hoá ở Silver.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["T_res"].value_counts().plot.bar(ax=axes[0], title="T_res (độ phân giải thời gian)")
df["case_definition_standardised"].value_counts().plot.barh(ax=axes[1], title="case_definition_standardised")
plt.tight_layout()

## 1.3 Độ phân giải thời gian có khớp độ dài khoảng thời gian không?

In [ ]:
# Week ~ 7 ngày, Month ~ 28-31, Year ~ 365/366 — kiểm tra có dòng lệch không
df.groupby("T_res")["period_days"].describe()

In [ ]:
odd = df[
    ((df.T_res == "Week") & (df.period_days != 7))
    | ((df.T_res == "Month") & ~df.period_days.between(28, 31))
    | ((df.T_res == "Year") & ~df.period_days.between(365, 366))
    | (df.period_days <= 0)
]
print(f"{len(odd)} dòng có độ dài khoảng thời gian không khớp T_res")
odd.head(10)

## 1.4 Phạm vi thời gian & độ phủ theo quốc gia

In [ ]:
print("Year:", df.Year.min(), "→", df.Year.max())
print("start_date:", df.calendar_start_date.min().date(), "→", df.calendar_start_date.max().date())
print("Số quốc gia:", df.ISO_A0.nunique())

fig, ax = plt.subplots(figsize=(14, 4))
df.groupby(["Year", "T_res"]).size().unstack(fill_value=0).plot.bar(stacked=True, ax=ax, width=0.9)
ax.set_title("Số dòng theo năm, tách theo T_res")
ax.set_xlabel("")
for i, label in enumerate(ax.get_xticklabels()):
    label.set_visible(i % 5 == 0)
plt.tight_layout()

In [ ]:
# Heatmap: quốc gia nào có dữ liệu năm nào (top 40 quốc gia nhiều dòng nhất)
top_c = df.ISO_A0.value_counts().head(40).index
cov = (df[df.ISO_A0.isin(top_c) & (df.Year >= 1990)]
       .groupby(["ISO_A0", "Year"]).size().unstack(fill_value=0))
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap((cov > 0).astype(int), cmap="Greens", cbar=False, ax=ax, linewidths=0.2)
ax.set_title("Độ phủ dữ liệu (xanh = có ít nhất 1 dòng), từ 1990")
plt.tight_layout()

In [ ]:
# Mỗi quốc gia: có bao nhiêu T_res khác nhau cùng một năm? (dấu hiệu dễ bị cộng trùng)
multi = (df.groupby(["ISO_A0", "Year"])["T_res"].nunique()
           .reset_index(name="n_T_res").query("n_T_res > 1"))
print(f"{len(multi)} cặp (quốc gia, năm) có >1 loại T_res")
multi.head(10)

> Nếu kết quả = 0: mỗi (quốc gia, năm) chỉ có **một** T_res → cộng theo năm an toàn.
> Nhưng cùng một quốc gia thì T_res **đổi theo giai đoạn** (vd. năm cũ chỉ có Year, gần đây có Week) — xem bảng ở mục 1.7.
> Code mục 1.6 vẫn chọn T_res ưu tiên cho chắc, phòng khi bản dữ liệu sau có chồng lấn.

## 1.5 Phân phối `dengue_total`

In [ ]:
print(df.dengue_total.describe().to_string())
print("Tỉ lệ dòng = 0:", round((df.dengue_total == 0).mean() * 100, 1), "%")
print("Giá trị âm:", (df.dengue_total < 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
np.log10(df.dengue_total[df.dengue_total > 0]).hist(bins=60, ax=axes[0])
axes[0].set_title("log10(dengue_total) — bỏ các dòng 0")
sns.boxplot(data=df[df.dengue_total > 0], x="T_res", y="dengue_total", ax=axes[1], log_scale=True)
axes[1].set_title("dengue_total theo T_res (log)")
plt.tight_layout()

In [ ]:
# Top outlier — kiểm tra xem có phải dữ liệu năm/cộng dồn không
df.nlargest(10, "dengue_total")[["full_name", "calendar_start_date", "calendar_end_date", "T_res",
                                 "case_definition_standardised", "dengue_total", "UUID"]]

## 1.6 Tổng số ca theo quốc gia (tránh cộng trùng)

In [ ]:
# Với mỗi (quốc gia, năm) chọn T_res ưu tiên: Week > Month > Year
prio = {"Week": 0, "Month": 1, "Year": 2}
tmp = df.assign(p=df.T_res.map(prio))
best = tmp.groupby(["ISO_A0", "Year"])["p"].transform("min")
dedup = tmp[tmp.p == best]

yearly = dedup.groupby(["ISO_A0", "full_name", "Year"], as_index=False)["dengue_total"].sum()
top15 = yearly.groupby("full_name")["dengue_total"].sum().nlargest(15)

fig, ax = plt.subplots(figsize=(10, 5))
top15.sort_values().plot.barh(ax=ax)
ax.set_title("Top 15 quốc gia theo tổng số ca (toàn giai đoạn, đã chọn 1 T_res/năm)")
plt.tight_layout()

In [ ]:
fig = px.line(yearly[yearly.full_name.isin(top15.index[:10]) & (yearly.Year >= 1990)],
              x="Year", y="dengue_total", color="full_name", log_y=True,
              title="Số ca theo năm — top 10 quốc gia (log scale)")
fig.show()

## 1.7 Đông Nam Á — chuỗi thời gian theo tuần

In [ ]:
SEA = ["VNM", "THA", "MYS", "SGP", "IDN", "PHL", "KHM", "LAO", "MMR", "BRN", "TLS"]
sea = df[df.ISO_A0.isin(SEA)]
print(sea.groupby(["full_name", "T_res"]).agg(
    rows=("dengue_total", "size"),
    first=("calendar_start_date", "min"),
    last=("calendar_start_date", "max"),
).to_string())

In [ ]:
sea_w = sea[sea.T_res == "Week"].sort_values("calendar_start_date")
fig = px.line(sea_w, x="calendar_start_date", y="dengue_total", color="full_name",
              title="Đông Nam Á — số ca theo tuần")
fig.show()

In [ ]:
# Mùa vụ: trung bình số ca theo tháng (từ dữ liệu tuần), chuẩn hoá theo từng nước
if len(sea_w):
    s = sea_w.assign(month=sea_w.calendar_start_date.dt.month)
    season = s.groupby(["full_name", "month"])["dengue_total"].mean().unstack()
    season = season.div(season.max(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(12, 0.6 * len(season) + 1.5))
    sns.heatmap(season, cmap="Reds", ax=ax, annot=True, fmt=".2f")
    ax.set_title("Tính mùa vụ (1.0 = tháng cao nhất của từng nước)")
    plt.tight_layout()

## 1.8 Kiểm tra trùng lặp & khoá

In [ ]:
key = ["ISO_A0", "calendar_start_date", "calendar_end_date", "case_definition_standardised"]
print("Dòng trùng hoàn toàn:", raw.duplicated().sum())
print("Dòng trùng theo khoá", key, ":", df.duplicated(key).sum())
print("Dòng trùng theo (ISO_A0, start, end):", df.duplicated(key[:3]).sum())

# Mỗi quốc gia ↔ tên có 1-1 không?
print("\nISO có nhiều tên:", (df.groupby("ISO_A0").full_name.nunique() > 1).sum())
print("ISO_A0 ≠ RNE_iso_code (số dòng):", (df.ISO_A0 != df.RNE_iso_code).sum())
df.loc[df.ISO_A0 != df.RNE_iso_code, ["full_name", "ISO_A0", "RNE_iso_code"]].drop_duplicates()

In [ ]:
dups = df[df.duplicated(key[:3], keep=False)].sort_values(key[:3])
dups[["full_name", "calendar_start_date", "calendar_end_date", "T_res",
      "case_definition_standardised", "dengue_total", "UUID"]].head(12)

> `RNE_iso_code` lệch với `ISO_A0` ở các vùng lãnh thổ hải ngoại (map về nước mẹ, vd. REU→FRA) và có vài giá trị trông như lỗi
> (GLP→`CLP`, TKL→`TKM` là Turkmenistan) → dùng **`ISO_A0`** làm khoá quốc gia.
>
> `UUID` là mã **nguồn dữ liệu** (mỗi nguồn nhiều dòng), không phải khoá dòng — cần tự tạo khoá ở Silver.

---
# Phần 2 — Google News RSS

Mỗi dòng jsonl = 1 bài báo: `title`, `link`, `pubDate`, `source`, `description` (HTML).

In [ ]:
frames = []
for f in news_files:
    part = pd.read_json(f, lines=True, dtype=False)
    part["_file"] = f.name
    frames.append(part)
news_raw = pd.concat(frames, ignore_index=True)
print(news_raw.shape)
print(news_raw.groupby("_file").size())
news_raw.head(3)

In [ ]:
# Metadata của feed (query nào sinh ra file này)
import xml.etree.ElementTree as ET
for f in xml_files:
    ch = ET.parse(f).getroot().find("channel")
    print(f.name, "|", ch.findtext("title"), "|", ch.findtext("lastBuildDate"),
          "|", len(ch.findall("item")), "items")

## 2.1 Null & trùng lặp giữa các lần ingest

In [ ]:
print(news_raw.drop(columns="_file").isna().sum().to_string())
print("\nlink unique:", news_raw.link.nunique(), "/", len(news_raw))
print("title unique:", news_raw.title.nunique())

# Các file có trùng nhau không?
sets = {f: set(g.link) for f, g in news_raw.groupby("_file")}
names = list(sets)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = sets[names[i]], sets[names[j]]
        print(f"{names[i]} ∩ {names[j]}: {len(a & b)} / {len(a | b)}")

> Chạy lại cùng query → gần như cùng một tập bài. Ở Silver cần **dedup theo `link`** (hoặc hash của title+source).

In [ ]:
news = news_raw.drop_duplicates("link").copy()
news["pub_ts"] = pd.to_datetime(news["pubDate"], utc=True, errors="coerce")

# Title của Google News có dạng "<headline> - <source>" → tách headline
news["headline"] = [t[: -len(" - " + s)] if isinstance(s, str) and t.endswith(" - " + s) else t
                    for t, s in zip(news.title, news.source)]
news["title_has_source_suffix"] = news.headline != news.title

# description là HTML: <a>headline</a>&nbsp;<font>source</font>
news["desc_text"] = news.description.map(lambda x: html.unescape(re.sub(r"<[^>]+>", " ", x or "")).strip())
news["desc_eq_title"] = [" ".join(d.split()).startswith(h[:40]) for d, h in zip(news.desc_text, news.headline)]

print("Số bài unique:", len(news))
print("pubDate parse lỗi:", news.pub_ts.isna().sum())
print("Title có đuôi ' - source':", news.title_has_source_suffix.mean().round(2))
print("description chỉ lặp lại title:", news.desc_eq_title.mean().round(2))
news[["headline", "source", "pub_ts", "desc_text"]].head(10)

> `description` gần như chỉ là title + source bọc HTML → **không có nội dung bài báo**. Muốn phân tích text sâu hơn phải crawl thêm (ngoài MVP).

## 2.2 Phân bố thời gian đăng bài

In [ ]:
print("pubDate:", news.pub_ts.min(), "→", news.pub_ts.max())
fig = px.histogram(news, x="pub_ts", nbins=40, title="Số bài theo thời gian đăng")
fig.show()

by_year = news.pub_ts.dt.year.value_counts().sort_index()
by_year.plot.bar(figsize=(10, 3), title="Số bài theo năm")
plt.tight_layout()

> RSS search của Google News trả về **bài liên quan nhất**, không phải mới nhất → trải dài nhiều năm.
> Nếu dùng làm tín hiệu "gần thời gian thực" thì cần lọc theo `pubDate` gần đây hoặc thêm `when:7d` vào query.

## 2.3 Nguồn báo

In [ ]:
src = news.source.value_counts()
print("Số nguồn:", len(src))
src.head(20).sort_values().plot.barh(figsize=(8, 6), title="Top 20 nguồn")
plt.tight_layout()

## 2.4 Nội dung headline

In [ ]:
news["title_len"] = news.headline.str.len()
news["title_words"] = news.headline.str.split().str.len()
print(news[["title_len", "title_words"]].describe().round(1).to_string())

In [ ]:
STOP = set("""a an the of in on for to and or with by at from as is are was be been after amid over into
new says say how what why its it this that than more up out s vs""".split())
words = Counter(
    w for t in news.headline.str.lower()
    for w in re.findall(r"[a-z]+", t) if w not in STOP and len(w) > 2
)
top_words = pd.Series(dict(words.most_common(25)))
top_words.sort_values().plot.barh(figsize=(8, 7), title="Top 25 từ trong headline")
plt.tight_layout()

In [ ]:
# Bài báo nhắc tới quốc gia nào? (đếm theo tên, đơn giản)
COUNTRIES = {
    "Vietnam": r"viet ?nam", "Thailand": r"thai", "Malaysia": r"malaysia", "Singapore": r"singapore",
    "Indonesia": r"indonesia", "Philippines": r"philippin|filipino", "Cambodia": r"cambodia",
    "Laos": r"\blao", "Myanmar": r"myanmar", "Bangladesh": r"bangladesh", "India": r"\bindia",
    "Brazil": r"brazil", "Sri Lanka": r"sri lanka",
}
text = (news.headline + " " + news.desc_text).str.lower()
mention = pd.Series({c: text.str.contains(p, regex=True).sum() for c, p in COUNTRIES.items()})
mention = mention[mention > 0].sort_values()
mention.plot.barh(figsize=(8, 4), title="Số bài nhắc tới quốc gia")
plt.tight_layout()
print("Bài không nhắc tới quốc gia nào trong list:",
      (~text.str.contains("|".join(COUNTRIES.values()), regex=True)).sum())

In [ ]:
# Có bao nhiêu bài thực sự về dengue?
kw = {"dengue": r"dengue", "outbreak": r"outbreak|surge|spike|rise|record", "death": r"death|die|dead|kill",
      "vaccine": r"vaccin", "mosquito": r"mosquito|aedes|wolbachia"}
pd.Series({k: text.str.contains(p).sum() for k, p in kw.items()}).sort_values().plot.barh(
    figsize=(7, 3), title="Chủ đề theo từ khoá")
plt.tight_layout()
news.loc[~text.str.contains("dengue"), ["headline", "source"]]

---
# Phần 3 — Hai nguồn có ghép được không?

Ý tưởng MVP: so sánh **số ca OpenDengue** với **số bài báo** theo (quốc gia, thời gian).

In [ ]:
news["year"] = news.pub_ts.dt.year
rows = []
for c, p in COUNTRIES.items():
    m = news[text.str.contains(p, regex=True)]
    for y, n in m.year.value_counts().items():
        rows.append({"full_name": c.upper(), "Year": y, "n_articles": n})
art = pd.DataFrame(rows)

cases = yearly.assign(full_name=yearly.full_name.str.upper())
joined = art.merge(cases, on=["full_name", "Year"], how="left")
print("Cặp (quốc gia, năm) có bài báo:", len(joined),
      "| khớp được số ca OpenDengue:", joined.dengue_total.notna().sum())
joined.sort_values(["full_name", "Year"])

> Tên quốc gia giữa 2 nguồn khác format (OpenDengue viết HOA, có thể khác tên như `VIET NAM`) → ở Silver nên map về **ISO3** làm khoá chung.

---
# Tóm tắt — việc cần làm ở Silver

**OpenDengue**
- Null encode bằng chuỗi `"NA"` → convert sang null thật.
- Bỏ các cột 100% null ở extract quốc gia (`adm_1_name`, `adm_2_name`, `IBGE_code`).
- Chuẩn hoá `case_definition_standardised` (`confirmed` vs `Confirmed`).
- Có 3 độ phân giải thời gian (Week/Month/Year) trộn lẫn; cùng một nước đổi T_res theo giai đoạn → giữ cột `T_res`, không vẽ/so sánh lẫn các độ phân giải.
- ~1/3 số dòng có `dengue_total = 0` → cần phân biệt "0 ca thật" với "không báo cáo" nếu được.
- Dùng `ISO_A0` làm khoá quốc gia (`RNE_iso_code` có giá trị sai).
- `UUID` là mã nguồn, không phải khoá dòng → khoá tự tạo: `(ISO_A0, calendar_start_date, calendar_end_date, case_definition)`.
- Dữ liệu chỉ tới 2025, dài từ 1924 → cân nhắc giới hạn khoảng năm cho MVP.

**Google News RSS**
- Mỗi lần ingest cùng query trả về gần như cùng tập bài → dedup theo `link`.
- `title` = `headline - source` → tách headline.
- `description` chỉ là HTML lặp lại title → không có thân bài.
- `pubDate` trải nhiều năm (sort theo relevance) → cần lọc thời gian / chỉnh query nếu muốn tín hiệu gần real-time.
- Chưa có cột quốc gia → cần trích xuất (regex/NER) + map ISO3 để join với OpenDengue.

*(Các nhận định trên dựa trên lần chạy hiện tại — kiểm tra lại số liệu output trong notebook khi có dữ liệu mới.)*